# 06 — SASRec for MerRec (FULL TRAIN sequences, RAM-safe, resumable)

- Scan **toàn bộ TRAIN event stream** và sắp theo thời gian.
- MerRec có ~27M TRAIN items nên không dùng embedding riêng cho toàn bộ 27M item theo cách naïve.
- Dùng **active-item vocabulary**; item ngoài vocab vẫn giữ trong sequence dưới token `OOV=1`.
- `PAD=0`, active item bắt đầu từ `sas_idx=2`.
- SASRec dùng sampled-negative objective, tránh full softmax hàng triệu item.
- VAL chọn checkpoint; TEST chỉ đánh giá cuối.
- TEST context = TRAIN + VAL lịch sử, nhưng không update trọng số bằng TEST.
- Có checkpoint/resume và FAISS IVF-PQ để serving.

Đầu ra deploy **không chỉ có trọng số**: gồm model weights + active item mapping + FAISS index + config + metrics.

In [ ]:
# Cell 1 — Imports / environment
from pathlib import Path
from datetime import datetime
import os, gc, json, math, time, random, shutil, zipfile

import numpy as np
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import psutil

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import faiss
except Exception:
    !pip -q install faiss-cpu
    import faiss

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device :", DEVICE)
if torch.cuda.is_available(): print("GPU    :", torch.cuda.get_device_name(0))

In [ ]:
# Verify the newly uploaded data and require a fresh training workspace.
from pathlib import Path
import polars as pl
import json
matches = list(Path('/kaggle/input').rglob('cf_train.parquet'))
assert len(matches) == 1, f'Expected one training dataset, found {len(matches)}'
md = matches[0].parent
expected = {'train': 2413351, 'val': 279422, 'test': 186393}
actual = {}
for split, count in expected.items():
    actual[split] = int(pl.scan_parquet(str(md / 'interactions' / ('split=' + split) / '*.parquet')).select((pl.col('event_group') == 'cart').sum()).collect(engine='streaming').item())
    assert actual[split] == count, f'Dataset mismatch: {split} cart={actual[split]}, expected {count}'
assert not Path('/kaggle/working/merrec_models').exists(), 'Fresh run required: old working model directory exists'
Path('/kaggle/working/data_verification.json').write_text(json.dumps({'cart_counts': actual, 'fresh_training': True}, indent=2))
print('New dataset verified; training from scratch:', actual)


In [ ]:
# Cell 2 — Configuration
# SASRec V2 — quality-first config for MerRec / Kaggle T4 x1

CFG = {
    "seed": SEED,

    # =========================================================
    # ITEM VOCABULARY
    # =========================================================
    # 10M active items ~82% TRAIN interaction coverage
    "min_item_interactions": 2,
    "max_active_items": 10_000_000,

    # =========================================================
    # SEQUENCE
    # =========================================================
    "max_seq_len": 50,

    # =========================================================
    # SASREC ARCHITECTURE
    # =========================================================
    "embed_dim": 64,
    "num_heads": 2,
    "num_blocks": 2,
    "dropout": 0.20,

    # =========================================================
    # TRAINING
    # =========================================================
    "batch_size": 512,

    # 4 negatives / positive thay vì 1
    "num_negatives": 4,

    # Không early-stop quá sớm
    "epochs": 12,
    "min_epochs": 4,
    "patience": 3,

    "lr_dense": 1e-3,
    "lr_item_embedding": 0.05,
    "weight_decay": 1e-5,
    "grad_clip": 1.0,

    # Save để resume khi Kaggle mất session
    "checkpoint_minutes": 15,

    # =========================================================
    # VALIDATION / MODEL SELECTION
    # =========================================================
    # Tăng từ 3k -> 5k để NDCG ổn định hơn
    "val_users": 5_000,

    # 200k hơi nhỏ khi vocab = 10M
    "val_candidate_items": 500_000,

    "eval_ks": [10, 20, 100],

    # Sau khi loại seen items vẫn còn đủ candidates
    "retrieve_extra": 500,

    # =========================================================
    # FINAL EVALUATION
    # =========================================================
    "full_eval_users": 10_000,

    # =========================================================
    # FAISS — 10M ITEM SERVING INDEX
    # =========================================================
    "faiss_train_items": 500_000,
    "faiss_nprobe": 64,
    "faiss_pq_m": 16,
    "faiss_pq_nbits": 8,
}

print(json.dumps(CFG, indent=2))

In [ ]:
# Cell 3 — Resolve paths + helpers
KINPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
ROOT=WORK/'merrec_models'/'SASRec'
for d in ['processed','checkpoints','model','index','mappings','metrics','status','serving']:
    (ROOT/d).mkdir(parents=True, exist_ok=True)

def find_one(name):
    ms=list(KINPUT.rglob(name)); ms2=[p for p in ms if 'merrec' in str(p).lower()]; ms=ms2 or ms
    if not ms: raise FileNotFoundError(name)
    return ms[0]

def schema_of(path_or_glob):
    s=str(path_or_glob)
    if '*' in s:
        gp=Path(s)
        fs=sorted(gp.parent.glob(gp.name))
        if not fs: raise FileNotFoundError(s)
        return pq.read_schema(fs[0]).names
    return pq.read_schema(path_or_glob).names

def pick(names,cands,required=True):
    m={x.lower():x for x in names}
    for c in cands:
        if c.lower() in m: return m[c.lower()]
    if required: raise KeyError((cands,names))
    return None

def atomic_json(o,p):
    p=Path(p); t=Path(str(p)+'.tmp'); t.write_text(json.dumps(o,indent=2,default=str),encoding='utf-8'); os.replace(t,p)

def atomic_torch(o,p):
    p=Path(p); t=Path(str(p)+'.tmp'); torch.save(o,t); os.replace(t,p)

def valid_parquet(p,n=1):
    try: return Path(p).exists() and pq.ParquetFile(p).metadata.num_rows>=n
    except: return False

def resources():
    vm=psutil.virtual_memory(); du=shutil.disk_usage(WORK)
    print(f"RAM available={vm.available/2**30:.2f}GB | working free={du.free/2**30:.2f}GB")

USER_MAP=find_one('train_user_map.parquet')
POP=find_one('popularity_train.parquet')
MODEL_DIR=USER_MAP.parent; INTDIR=MODEL_DIR/'interactions'
TRAIN_GLOB=str(INTDIR/'split=train'/'*.parquet'); VAL_GLOB=str(INTDIR/'split=val'/'*.parquet'); TEST_GLOB=str(INTDIR/'split=test'/'*.parquet')

# Fresh training: do not restore previous input checkpoints.
print('USER_MAP:',USER_MAP); print('POP:',POP); print('INTDIR:',INTDIR); print('ROOT:',ROOT); resources()

In [ ]:
# Cell 4 — Build ACTIVE item vocabulary (stable version)
#
# PAD = 0
# OOV = 1
# Active items start from sas_idx = 2
#
# FIX:
# Không dùng LazyFrame.sink_parquet() cho unique -> sort -> head
# vì Polars có thể panic ở top_k optimizer.
# Ta collect 2 cột cần thiết trước, sau đó xử lý eager.

ACTIVE_MAP = ROOT / "mappings" / "active_item_map.parquet"
ACTIVE_OK = ROOT / "status" / "_ACTIVE_MAP_SUCCESS.json"

# ------------------------------------------------------------
# Population schema
# ------------------------------------------------------------
pop_cols = schema_of(POP)

POP_ITEM = pick(
    pop_cols,
    ["item_id", "itemid", "item"]
)

POP_FREQ = pick(
    pop_cols,
    ["interactions", "count", "n_interactions"]
)

POP_SCORE = pick(
    pop_cols,
    ["popularity_score", "score"],
    required=False
)

print("POP_ITEM :", POP_ITEM)
print("POP_FREQ :", POP_FREQ)
print("POP_SCORE:", POP_SCORE)


# ------------------------------------------------------------
# Reuse if already complete
# ------------------------------------------------------------
if (
    valid_parquet(ACTIVE_MAP, 1000)
    and ACTIVE_OK.exists()
):

    info = json.loads(
        ACTIVE_OK.read_text()
    )

    N_ACTIVE = int(
        info["n_active_items"]
    )

    print(
        f"✅ Reuse ACTIVE MAP: "
        f"{N_ACTIVE:,} items"
    )

else:

    print(
        "🚀 Building ACTIVE item vocabulary..."
    )

    # --------------------------------------------------------
    # STEP 1 — Read only the required columns
    # --------------------------------------------------------
    select_exprs = [
        pl.col(POP_ITEM)
        .alias("item_id"),

        pl.col(POP_FREQ)
        .cast(pl.UInt64)
        .fill_null(0)
        .alias("interactions"),
    ]

    if POP_SCORE is not None:
        select_exprs.append(
            pl.col(POP_SCORE)
            .cast(pl.Float64)
            .fill_null(0.0)
            .alias("_score")
        )
    else:
        select_exprs.append(
            pl.col(POP_FREQ)
            .cast(pl.Float64)
            .fill_null(0.0)
            .alias("_score")
        )

    print("Reading popularity table...")

    pop_df = (
        pl.scan_parquet(POP)
        .select(select_exprs)
        .collect(
            engine="streaming"
        )
    )

    print(
        f"Loaded popularity rows: "
        f"{pop_df.height:,}"
    )

    # --------------------------------------------------------
    # STEP 2 — Remove duplicated item_id
    # --------------------------------------------------------
    pop_df = (
        pop_df
        .unique(
            subset=["item_id"],
            keep="last"
        )
    )

    print(
        f"Unique items: "
        f"{pop_df.height:,}"
    )

    # --------------------------------------------------------
    # STEP 3 — Filter by minimum interactions
    # --------------------------------------------------------
    pop_df = (
        pop_df
        .filter(
            pl.col("interactions")
            >= int(
                CFG["min_item_interactions"]
            )
        )
    )

    print(
        f"After min frequency filter: "
        f"{pop_df.height:,}"
    )

    # --------------------------------------------------------
    # STEP 4 — Sort eagerly
    # --------------------------------------------------------
    pop_df = (
        pop_df
        .sort(
            by=[
                "interactions",
                "_score",
                "item_id",
            ],
            descending=[
                True,
                True,
                False,
            ],
        )
    )

    # --------------------------------------------------------
    # STEP 5 — Take top K
    # --------------------------------------------------------
    max_active = int(
        CFG["max_active_items"]
    )

    pop_df = pop_df.head(
        max_active
    )

    # --------------------------------------------------------
    # STEP 6 — Add SASRec indices
    #
    # PAD = 0
    # OOV = 1
    # Active starts at 2
    # --------------------------------------------------------
    active_df = (
        pop_df
        .with_row_index(
            name="sas_idx",
            offset=2,
        )
        .with_columns(
            pl.col("sas_idx")
            .cast(pl.UInt32)
        )
        .select([
            "item_id",
            "sas_idx",
            "interactions",
        ])
    )

    # --------------------------------------------------------
    # STEP 7 — Safe eager write
    # --------------------------------------------------------
    tmp = Path(
        str(ACTIVE_MAP)
        + ".tmp.parquet"
    )

    if tmp.exists():
        tmp.unlink()

    active_df.write_parquet(
        tmp,
        compression="zstd",
        row_group_size=250_000,
    )

    os.replace(
        tmp,
        ACTIVE_MAP
    )

    N_ACTIVE = int(
        active_df.height
    )

    # --------------------------------------------------------
    # STEP 8 — Save completion marker
    # --------------------------------------------------------
    atomic_json(
        {
            "n_active_items": N_ACTIVE,
            "min_item_interactions":
                int(
                    CFG[
                        "min_item_interactions"
                    ]
                ),
            "max_active_items":
                int(
                    CFG[
                        "max_active_items"
                    ]
                ),
            "PAD": 0,
            "OOV": 1,
            "created_at":
                datetime.now().isoformat(),
        },
        ACTIVE_OK,
    )

    del active_df
    del pop_df

    gc.collect()

    print(
        f"✅ ACTIVE MAP ready: "
        f"{N_ACTIVE:,} items"
    )


VOCAB_SIZE = (
    int(N_ACTIVE)
    + 2
)

print(
    f"Vocabulary size: "
    f"{VOCAB_SIZE:,}"
)

resources()

In [ ]:
# Cell 5 — Encode and chronological-sort ALL TRAIN events
SORTED_EVENTS=ROOT/'processed'/'train_events_sorted.parquet'; SORTED_OK=ROOT/'status'/'_SORTED_EVENTS_SUCCESS.json'
train_cols=schema_of(TRAIN_GLOB)
TR_USER=pick(train_cols,['user_id','userid','user']); TR_ITEM=pick(train_cols,['item_id','itemid','item']); TR_TS=pick(train_cols,['ts','timestamp','event_time','stime'])
user_cols=schema_of(USER_MAP); UM_USER=pick(user_cols,['user_id','userid','user']); UM_IDX=pick(user_cols,['user_idx','user_index','uid','idx'])

if not (valid_parquet(SORTED_EVENTS,1000) and SORTED_OK.exists()):
    users=pl.scan_parquet(USER_MAP).select([pl.col(UM_USER).cast(pl.Utf8).alias('user_id'),pl.col(UM_IDX).cast(pl.UInt32).alias('user_idx')])
    active=pl.scan_parquet(ACTIVE_MAP).select([pl.col('item_id').cast(pl.Utf8),pl.col('sas_idx').cast(pl.UInt32)])
    events=(pl.scan_parquet(TRAIN_GLOB)
        .select([pl.col(TR_USER).cast(pl.Utf8).alias('user_id'),pl.col(TR_ITEM).cast(pl.Utf8).alias('item_id'),pl.col(TR_TS).alias('ts')])
        .join(users,on='user_id',how='inner').join(active,on='item_id',how='left')
        .with_columns(pl.col('sas_idx').fill_null(1).cast(pl.UInt32))
        .sort(['user_idx','ts']).select(['user_idx','sas_idx']))
    tmp=Path(str(SORTED_EVENTS)+'.tmp');
    if tmp.exists(): tmp.unlink()
    events.sink_parquet(tmp,compression='zstd',row_group_size=1_000_000,maintain_order=True); os.replace(tmp,SORTED_EVENTS)
    pf=pq.ParquetFile(SORTED_EVENTS); total=pf.metadata.num_rows; active_rows=0; seen=0
    for b in pf.iter_batches(batch_size=2_000_000,columns=['sas_idx']):
        a=b.column('sas_idx').to_numpy(zero_copy_only=False); seen+=len(a); active_rows+=int((a>1).sum())
    atomic_json({'train_events':int(total),'active_item_events':int(active_rows),'active_event_coverage':float(active_rows/max(1,seen))},SORTED_OK)

st=json.loads(SORTED_OK.read_text()); print(json.dumps(st,indent=2)); resources()

In [ ]:
# Cell 6 — Build sequence windows
SEQ_FILE=ROOT/'processed'/'train_sequence_windows.parquet'; SEQ_OK=ROOT/'status'/'_SEQUENCES_SUCCESS.json'; L=int(CFG['max_seq_len'])

def build_sequence_windows():
    if valid_parquet(SEQ_FILE,1000) and SEQ_OK.exists(): return
    if SEQ_FILE.exists(): SEQ_FILE.unlink()
    pf=pq.ParquetFile(SORTED_EVENTS)
    schema=pa.schema([('user_idx',pa.uint32()),('seq',pa.list_(pa.uint32()))])
    writer=pq.ParquetWriter(SEQ_FILE,schema=schema,compression='zstd')
    buf_u=[]; buf_s=[]; n_users=n_windows=n_transitions=n_active_targets=0
    current_user=None; current_seq=[]

    def flush():
        nonlocal buf_u,buf_s
        if not buf_u: return
        writer.write_table(pa.table({'user_idx':pa.array(buf_u,type=pa.uint32()),'seq':pa.array(buf_s,type=pa.list_(pa.uint32()))}),row_group_size=50000)
        buf_u=[]; buf_s=[]

    def emit(uid,seq):
        nonlocal n_users,n_windows,n_transitions,n_active_targets
        if uid is None or len(seq)<2: return
        n_users+=1
        for start in range(0,len(seq)-1,L):
            seg=seq[start:start+L+1]
            if len(seg)<2: continue
            buf_u.append(int(uid)); buf_s.append([int(x) for x in seg]); n_windows+=1
            t=np.asarray(seg[1:],dtype=np.int64); n_transitions+=len(t); n_active_targets+=int((t>1).sum())
            if len(buf_u)>=50000: flush()

    processed=0
    for batch in pf.iter_batches(batch_size=1_000_000,columns=['user_idx','sas_idx']):
        us=batch.column('user_idx').to_numpy(zero_copy_only=False); its=batch.column('sas_idx').to_numpy(zero_copy_only=False)
        for u,it in zip(us,its):
            u=int(u); it=int(it)
            if current_user is None: current_user=u
            if u!=current_user:
                emit(current_user,current_seq); current_user=u; current_seq=[]
            current_seq.append(it)
        processed+=len(us)
        if processed%10_000_000<1_000_000: print(f'processed events: {processed:,}')
    emit(current_user,current_seq); flush(); writer.close()
    atomic_json({'users_with_sequence':n_users,'windows':n_windows,'transitions':n_transitions,'active_target_transitions':n_active_targets,'active_target_coverage':n_active_targets/max(1,n_transitions),'max_seq_len':L},SEQ_OK)

build_sequence_windows(); print(json.dumps(json.loads(SEQ_OK.read_text()),indent=2)); resources()

In [ ]:
# Cell 7 — Evaluation bundles
ACTIVE_LAZY=pl.scan_parquet(ACTIVE_MAP).select([pl.col('item_id').cast(pl.Utf8),pl.col('sas_idx').cast(pl.UInt32)])
USER_LAZY=pl.scan_parquet(USER_MAP).select([pl.col(UM_USER).cast(pl.Utf8).alias('user_id'),pl.col(UM_IDX).cast(pl.UInt32).alias('user_idx')])
POSITIVE_EVENTS={'like','cart','offer','buy_start','buy_comp'}; STRONG_EVENTS={'cart','offer','buy_start','buy_comp'}; PURCHASE_EVENTS={'buy_comp'}

def split_glob(split): return str(INTDIR/f'split={split}'/'*.parquet')

def history_lazy(splits):
    fs=[]
    for sp in splits:
        g=split_glob(sp); cols=schema_of(g); u=pick(cols,['user_id','userid','user']); i=pick(cols,['item_id','itemid','item']); t=pick(cols,['ts','timestamp','event_time','stime'])
        fs.append(pl.scan_parquet(g).select([pl.col(u).cast(pl.Utf8).alias('user_id'),pl.col(i).cast(pl.Utf8).alias('item_id'),pl.col(t).alias('ts')]))
    return pl.concat(fs,how='vertical_relaxed')

def future_lazy(split):
    g=split_glob(split); cols=schema_of(g); u=pick(cols,['user_id','userid','user']); i=pick(cols,['item_id','itemid','item']); e=pick(cols,['event_group','event','event_type','action'],False); st=pick(cols,['is_strong_positive'],False); pu=pick(cols,['is_purchase'],False)
    ex=[pl.col(u).cast(pl.Utf8).alias('user_id'),pl.col(i).cast(pl.Utf8).alias('item_id')]
    ex.append(pl.col(e).cast(pl.Utf8).str.to_lowercase().alias('event_group') if e else pl.lit('').alias('event_group'))
    ex.append(pl.col(st).cast(pl.UInt8).alias('is_strong_positive') if st else pl.lit(0,dtype=pl.UInt8).alias('is_strong_positive'))
    ex.append(pl.col(pu).cast(pl.UInt8).alias('is_purchase') if pu else pl.lit(0,dtype=pl.UInt8).alias('is_purchase'))
    return pl.scan_parquet(g).select(ex)

def flags(ev,strong,purchase):
    ev=str(ev or '').lower(); return {'ALL':True,'POSITIVE':ev in POSITIVE_EVENTS,'STRONG':bool(strong) or ev in STRONG_EVENTS,'PURCHASE':bool(purchase) or ev in PURCHASE_EVENTS}

def build_eval_bundle(split,max_users,history_splits):
    fut=future_lazy(split); cand=(fut.select('user_id').unique().with_columns(pl.col('user_id').hash(seed=SEED).alias('_h')).sort('_h').head(max(max_users*20,max_users)).collect(engine='streaming'))
    cohort=(cand.lazy().join(USER_LAZY,on='user_id',how='inner').sort('_h').head(max_users).select(['user_id','user_idx']).collect(engine='streaming'))
    if cohort.height==0: raise RuntimeError(f'No warm users for {split}')
    hist=(history_lazy(history_splits).join(cohort.lazy(),on='user_id',how='inner').join(ACTIVE_LAZY,on='item_id',how='left').with_columns(pl.col('sas_idx').fill_null(1).cast(pl.UInt32)).sort(['user_idx','ts']).select(['user_idx','sas_idx']).collect(engine='streaming'))
    contexts={}; seen={}
    for u,it in hist.iter_rows():
        u=int(u); it=int(it); contexts.setdefault(u,[]).append(it)
        if it>1: seen.setdefault(u,set()).add(it)
    for u in list(contexts): contexts[u]=contexts[u][-L:]
    future=(fut.join(cohort.lazy(),on='user_id',how='inner').join(ACTIVE_LAZY,on='item_id',how='left').collect(engine='streaming'))
    raw={m:{} for m in ['ALL','POSITIVE','STRONG','PURCHASE']}; warm={m:{} for m in raw}
    for r in future.iter_rows(named=True):
        u=int(r['user_idx']); raw_item=str(r['item_id']); sas=r['sas_idx']
        for m,yes in flags(r['event_group'],r['is_strong_positive'],r['is_purchase']).items():
            if not yes: continue
            raw[m].setdefault(u,set()).add(raw_item)
            if sas is not None and int(sas)>1: warm[m].setdefault(u,set()).add(int(sas))
    users=[int(u) for u in cohort['user_idx'].to_list() if int(u) in contexts and contexts[int(u)]]
    return {'split':split,'users':users,'contexts':contexts,'seen':seen,'targets':warm,'targets_all':{m:sum(len(s) for s in raw[m].values()) for m in raw},'targets_warm':{m:sum(len(s) for s in warm[m].values()) for m in warm}}

VAL_BUNDLE=build_eval_bundle('val',CFG['val_users'],['train'])
print('VAL users:',len(VAL_BUNDLE['users']))
for m in ['ALL','POSITIVE','STRONG','PURCHASE']:
    a=VAL_BUNDLE['targets_all'][m]; w=VAL_BUNDLE['targets_warm'][m]; print(m,'all=',a,'active=',w,'coverage=',w/a if a else 0)
resources()

In [ ]:
# Cell 8 — SASRec model + optimizers + resume
class SASRec(nn.Module):
    def __init__(self,vocab_size,max_len,d_model,n_heads,n_blocks,dropout):
        super().__init__(); self.max_len=max_len; self.d_model=d_model
        self.item_emb=nn.Embedding(vocab_size,d_model,padding_idx=0,sparse=True)
        self.pos_emb=nn.Embedding(max_len,d_model); self.emb_dropout=nn.Dropout(dropout)
        layer=nn.TransformerEncoderLayer(d_model=d_model,nhead=n_heads,dim_feedforward=d_model*4,dropout=dropout,activation='gelu',batch_first=True,norm_first=True)
        self.encoder=nn.TransformerEncoder(layer,num_layers=n_blocks); self.final_norm=nn.LayerNorm(d_model)
        self.register_buffer('causal_mask',torch.triu(torch.ones(max_len,max_len,dtype=torch.bool),diagonal=1),persistent=False)
        nn.init.normal_(self.item_emb.weight,std=.02); nn.init.normal_(self.pos_emb.weight,std=.02)
        with torch.no_grad(): self.item_emb.weight[0].zero_()
    def encode_all(self,seq):
        B,Lx=seq.shape; pos=torch.arange(Lx,device=seq.device).unsqueeze(0).expand(B,-1); pad=seq.eq(0)
        x=self.item_emb(seq)*math.sqrt(self.d_model)+self.pos_emb(pos); x=self.emb_dropout(x)
        x=self.encoder(x,mask=self.causal_mask[:Lx,:Lx],src_key_padding_mask=pad); x=self.final_norm(x); return x.masked_fill(pad.unsqueeze(-1),0.)
    def training_logits(self,seq,pos_items,neg_items):
        h=self.encode_all(seq); return (h*self.item_emb(pos_items)).sum(-1),(h*self.item_emb(neg_items)).sum(-1)
    def encode_context(self,seq): return self.encode_all(seq)[:,-1,:]

model=SASRec(VOCAB_SIZE,CFG['max_seq_len'],CFG['embed_dim'],CFG['num_heads'],CFG['num_blocks'],CFG['dropout']).to(DEVICE)
dense_params=[p for n,p in model.named_parameters() if n!='item_emb.weight']
opt_sparse=torch.optim.SGD([model.item_emb.weight],lr=CFG['lr_item_embedding'])
opt_dense=torch.optim.AdamW(dense_params,lr=CFG['lr_dense'],weight_decay=CFG['weight_decay'])
LATEST=ROOT/'checkpoints'/'sasrec_latest.pt'; BEST=ROOT/'checkpoints'/'sasrec_best.pt'
epoch0=rg0=step=0; best_metric=-1.; bad=0; history=[]
if LATEST.exists():
    ck=torch.load(LATEST,map_location=DEVICE,weights_only=False); model.load_state_dict(ck['model']); opt_sparse.load_state_dict(ck['opt_sparse']); opt_dense.load_state_dict(ck['opt_dense'])
    epoch0=int(ck['epoch']); rg0=int(ck['rg']); step=int(ck['step']); best_metric=float(ck['best_metric']); bad=int(ck['bad']); history=ck.get('history',[]); print('RESUME:',epoch0,rg0,step,'best=',best_metric)

def save_latest(epoch,rg):
    atomic_torch({'model':model.state_dict(),'opt_sparse':opt_sparse.state_dict(),'opt_dense':opt_dense.state_dict(),'epoch':int(epoch),'rg':int(rg),'step':int(step),'best_metric':float(best_metric),'bad':int(bad),'history':history,'cfg':CFG},LATEST)

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}'); print('Item embedding GB:',model.item_emb.weight.numel()*4/2**30); resources()

In [ ]:
# Cell 9 — Batching + evaluation helpers
def make_train_batch(seq_list,rng):
    B=len(seq_list); x=np.zeros((B,L),np.int64); p=np.zeros((B,L),np.int64)
    for r,seq in enumerate(seq_list):
        a=np.asarray(seq,dtype=np.int64)
        if len(a)<2: continue
        inp=a[:-1][-L:]; pos=a[1:][-L:]; x[r,-len(inp):]=inp; p[r,-len(pos):]=pos
    n=rng.integers(2,VOCAB_SIZE,size=p.shape,dtype=np.int64); same=(n==p)&(p>1)
    while same.any():
        n[same]=rng.integers(2,VOCAB_SIZE,size=int(same.sum()),dtype=np.int64); same=(n==p)&(p>1)
    return torch.from_numpy(x).to(DEVICE),torch.from_numpy(p).to(DEVICE),torch.from_numpy(n).to(DEVICE)

def contexts_to_tensor(bundle,users):
    x=np.zeros((len(users),L),np.int64)
    for r,u in enumerate(users):
        seq=np.asarray(bundle['contexts'][int(u)][-L:],dtype=np.int64); x[r,-len(seq):]=seq
    return torch.from_numpy(x).to(DEVICE)

def encode_queries(bundle,users,batch_size=512):
    model.eval(); out=[]
    with torch.no_grad():
        for s in range(0,len(users),batch_size): out.append(model.encode_context(contexts_to_tensor(bundle,users[s:s+batch_size])).cpu().numpy().astype(np.float32))
    return np.concatenate(out)

def item_vectors(ids,batch_size=100000):
    model.eval(); ids=np.asarray(ids,dtype=np.int64); out=[]
    with torch.no_grad():
        for s in range(0,len(ids),batch_size):
            t=torch.from_numpy(ids[s:s+batch_size]).to(DEVICE); out.append(model.item_emb(t).cpu().numpy().astype(np.float32))
    return np.concatenate(out)

def ndcg(pred,t,k):
    dc=sum(1/math.log2(r+1) for r,x in enumerate(pred[:k],1) if x in t); idc=sum(1/math.log2(r+1) for r in range(1,min(len(t),k)+1)); return dc/idc if idc else 0.

def evaluate_predictions(bundle,recs):
    O={}
    for m in ['ALL','POSITIVE','STRONG','PURCHASE']:
        tmap=bundle['targets'][m]; a=bundle['targets_all'][m]; w=bundle['targets_warm'][m]; row={'targets_all':int(a),'targets_active':int(w),'active_target_coverage':w/max(1,a)}
        for k in CFG['eval_ks']:
            rr=[]; pp=[]; hh=[]; nn=[]
            for u in bundle['users']:
                t=tmap.get(int(u),set())
                if not t: continue
                pr=recs.get(int(u),[])[:k]; hit=sum(x in t for x in pr); rr.append(hit/len(t)); pp.append(hit/k); hh.append(float(hit>0)); nn.append(ndcg(pr,t,k))
            row[f'Recall@{k}']=float(np.mean(rr)) if rr else 0.; row[f'Precision@{k}']=float(np.mean(pp)) if pp else 0.; row[f'HitRate@{k}']=float(np.mean(hh)) if hh else 0.; row[f'NDCG@{k}']=float(np.mean(nn)) if nn else 0.
        O[m]=row
    return O

def search_index(index,bundle):
    users=bundle['users']; q=encode_queries(bundle,users); maxk=max(CFG['eval_ks']); _,ids=index.search(q,maxk+CFG['retrieve_extra']); recs={}
    for r,u in enumerate(users):
        banned=bundle['seen'].get(int(u),set()); z=[]
        for x in ids[r]:
            x=int(x)
            if x>1 and x not in banned: z.append(x)
            if len(z)>=maxk: break
        recs[int(u)]=z
    return evaluate_predictions(bundle,recs)

rng=np.random.default_rng(SEED+91); n_rand=min(CFG['val_candidate_items'],N_ACTIVE); random_ids=rng.choice(np.arange(2,VOCAB_SIZE,dtype=np.int64),size=n_rand,replace=False)
gt=[]
for mm in VAL_BUNDLE['targets'].values():
    for ss in mm.values(): gt.extend(ss)
VAL_CANDIDATES=np.unique(np.concatenate([random_ids,np.asarray(gt,dtype=np.int64) if gt else np.empty(0,dtype=np.int64)])); print('VAL candidates:',len(VAL_CANDIDATES))

In [ ]:
# Cell 10 — Approximate VAL model-selection metric
def build_candidate_index(ids):
    ids=np.asarray(ids,dtype=np.int64); vec=item_vectors(ids)
    if len(ids)<50000:
        ix=faiss.IndexIDMap2(faiss.IndexFlatIP(CFG['embed_dim'])); ix.add_with_ids(vec,ids); return ix
    nlist=min(512,max(64,int(math.sqrt(len(ids))))); ix=faiss.IndexIVFFlat(faiss.IndexFlatIP(CFG['embed_dim']),CFG['embed_dim'],nlist,faiss.METRIC_INNER_PRODUCT)
    train_n=min(len(vec),max(50000,nlist*40)); si=np.random.default_rng(SEED+123).choice(len(vec),size=train_n,replace=False); ix.train(vec[si]); ix.add_with_ids(vec,ids); ix.nprobe=min(24,nlist); return ix

def val_metric():
    ix=build_candidate_index(VAL_CANDIDATES); m=search_index(ix,VAL_BUNDLE); del ix; gc.collect(); return m
print('VAL selection helper ready')

In [ ]:
# Cell 11 — TRAIN + checkpoint + VAL selection
seq_pf=pq.ParquetFile(SEQ_FILE); N_RG=seq_pf.num_row_groups; print('Sequence row groups:',N_RG)
for ep in range(epoch0,CFG['epochs']):
    model.train(); order=np.random.default_rng(SEED+ep).permutation(N_RG); start=rg0 if ep==epoch0 else 0; loss_sum=0.; nb=0; active_positions=0; last=time.time()
    print('='*90); print(f"EPOCH {ep+1} / {CFG['epochs']} | resume_rg={start}")
    for pos in range(start,N_RG):
        rg=int(order[pos]); table=seq_pf.read_row_group(rg,columns=['seq']); seqs=table.column('seq').to_pylist(); perm=np.random.default_rng(SEED+ep*1_000_003+rg).permutation(len(seqs))
        for s in range(0,len(perm),CFG['batch_size']):
            idx=perm[s:s+CFG['batch_size']]; batch=[seqs[int(i)] for i in idx]; x,p,n=make_train_batch(batch,np.random.default_rng(SEED+step*17+ep)); mask=p.gt(1); nvalid=int(mask.sum().item())
            if nvalid==0: continue
            plog,nlog=model.training_logits(x,p,n); loss=(-F.logsigmoid(plog)-F.logsigmoid(-nlog))[mask].mean()
            opt_sparse.zero_grad(set_to_none=True); opt_dense.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(dense_params,CFG['grad_clip']); opt_sparse.step(); opt_dense.step()
            with torch.no_grad(): model.item_emb.weight[0].zero_()
            loss_sum+=float(loss.detach().cpu()); nb+=1; active_positions+=nvalid; step+=1
        if (pos+1)%5==0 or pos+1==N_RG: print(f'rg {pos+1}/{N_RG} step={step:,} loss={loss_sum/max(1,nb):.5f} active_targets={active_positions:,}')
        if time.time()-last>=CFG['checkpoint_minutes']*60: save_latest(ep,pos+1); print('✅ checkpoint',pos+1,'/',N_RG); resources(); last=time.time()
        del table,seqs; gc.collect()
    vm=val_metric(); score=float(vm['POSITIVE']['NDCG@20']); history.append({'epoch':ep+1,'loss':loss_sum/max(1,nb),'active_target_positions':active_positions,'val_selection_metric':'POSITIVE_NDCG@20','val_selection_score':score,'val':vm}); atomic_json(history,ROOT/'metrics'/'train_history.json')
    print(json.dumps(vm,indent=2)); print('VAL POSITIVE NDCG@20 =',score)
    if score>best_metric:
        best_metric=score; bad=0; atomic_torch({'model':model.state_dict(),'epoch':ep+1,'score':score,'cfg':CFG,'vocab_size':VOCAB_SIZE},BEST); print('🏆 NEW BEST',score)
    else: bad+=1; print(f"No improvement {bad}/{CFG['patience']}")
    rg0=0; save_latest(ep+1,0)
    if bad>=CFG['patience']: print('Early stopping'); break
print('✅ SASRec TRAINING COMPLETE')

In [ ]:
# Cell 12 — Load BEST + deployable model weights
if not BEST.exists(): raise FileNotFoundError(BEST)
best_ck=torch.load(BEST,map_location=DEVICE,weights_only=False); model.load_state_dict(best_ck['model']); model.eval()
FINAL_MODEL=ROOT/'model'/'sasrec_best_model.pt'
atomic_torch({'model':model.state_dict(),'cfg':CFG,'vocab_size':VOCAB_SIZE,'n_active_items':N_ACTIVE,'PAD':0,'OOV':1,'best_epoch':int(best_ck['epoch']),'best_val_positive_ndcg20':float(best_ck['score'])},FINAL_MODEL)
print('Best epoch:',best_ck['epoch']); print('Best VAL POSITIVE NDCG@20:',best_ck['score']); print('Saved:',FINAL_MODEL); resources()

In [ ]:
# Cell 13 — FULL active-item FAISS IVF-PQ
FAISS_PATH=ROOT/'index'/'items_ivfpq.faiss'; FAISS_OK=ROOT/'status'/'_FAISS_SUCCESS.json'
if not (FAISS_PATH.exists() and FAISS_OK.exists()):
    D=CFG['embed_dim']; M=CFG['faiss_pq_m']; assert D%M==0
    nlist=min(4096,max(256,int(2*math.sqrt(N_ACTIVE)))); index=faiss.IndexIVFPQ(faiss.IndexFlatIP(D),D,nlist,M,CFG['faiss_pq_nbits'],faiss.METRIC_INNER_PRODUCT)
    rng=np.random.default_rng(SEED+500); train_n=min(N_ACTIVE,max(CFG['faiss_train_items'],nlist*40)); train_ids=rng.choice(np.arange(2,VOCAB_SIZE,dtype=np.int64),size=train_n,replace=False)
    print(f'Training FAISS nlist={nlist}, items={train_n:,}'); tv=item_vectors(train_ids); index.train(tv); del tv; gc.collect()
    bs=100000
    for start in range(2,VOCAB_SIZE,bs):
        ids=np.arange(start,min(start+bs,VOCAB_SIZE),dtype=np.int64); vec=item_vectors(ids); index.add_with_ids(vec,ids); done=min(start+bs,VOCAB_SIZE)-2
        if done%500000<bs: print(f'FAISS added {done:,}/{N_ACTIVE:,}')
        del vec; gc.collect()
    index.nprobe=CFG['faiss_nprobe']; faiss.write_index(index,str(FAISS_PATH)); atomic_json({'index_type':'IVF-PQ','metric':'inner_product','n_active_items':N_ACTIVE,'nlist':nlist,'nprobe':CFG['faiss_nprobe'],'pq_m':M,'pq_nbits':CFG['faiss_pq_nbits']},FAISS_OK)
else:
    index=faiss.read_index(str(FAISS_PATH)); index.nprobe=CFG['faiss_nprobe']
print('FAISS:',FAISS_PATH); print('ntotal:',index.ntotal); resources()

In [ ]:
# Cell 14 — FINAL VAL + TEST evaluation
VAL_FULL=build_eval_bundle('val',CFG['full_eval_users'],['train'])
TEST_FULL=build_eval_bundle('test',CFG['full_eval_users'],['train','val'])
val_metrics=search_index(index,VAL_FULL); test_metrics=search_index(index,TEST_FULL)
atomic_json(val_metrics,ROOT/'metrics'/'val_full_index_metrics.json'); atomic_json(test_metrics,ROOT/'metrics'/'test_full_index_metrics.json')
coverage={'active_vocab_items':N_ACTIVE,'vocab_size_with_special_tokens':VOCAB_SIZE,'train_event_active_coverage':json.loads(SORTED_OK.read_text())['active_event_coverage'],'train_active_target_coverage':json.loads(SEQ_OK.read_text())['active_target_coverage'],'val_target_coverage':{m:val_metrics[m]['active_target_coverage'] for m in val_metrics},'test_target_coverage':{m:test_metrics[m]['active_target_coverage'] for m in test_metrics}}
atomic_json(coverage,ROOT/'metrics'/'coverage_summary.json'); atomic_json({'model':'SASRec','best_epoch':int(best_ck['epoch']),'best_val_selection_positive_ndcg20':float(best_ck['score']),'val':val_metrics,'test':test_metrics},ROOT/'metrics'/'summary.json')
print('VAL FULL'); print(json.dumps(val_metrics,indent=2)); print(); print('TEST FULL'); print(json.dumps(test_metrics,indent=2)); print(); print('COVERAGE'); print(json.dumps(coverage,indent=2))

In [ ]:
# Cell 15 — Serving package ZIP
SERVING_CFG=ROOT/'serving'/'sasrec_config.json'; README=ROOT/'serving'/'README_SERVING.txt'
atomic_json({'model_type':'SASRec','objective':'sampled-negative sequential next-item prediction','similarity':'inner_product','PAD':0,'OOV':1,'active_item_vocabulary':N_ACTIVE,'max_seq_len':CFG['max_seq_len'],'test_context':'TRAIN + VAL history','cfg':CFG},SERVING_CFG)
README.write_text(chr(10).join([
    'MerRec SASRec serving package','',
    'Required:','1) model/sasrec_best_model.pt','2) mappings/active_item_map.parquet','3) index/items_ivfpq.faiss','4) sasrec_config.json','',
    'Flow: recent item_ids -> active_item_map -> inactive=>OOV=1 -> last max_seq_len -> SASRec -> FAISS -> sas_idx -> item_id -> catalog/database','',
    'PAD=0, OOV=1, active items start at sas_idx=2.','VAL selects checkpoint; TEST is final only. TEST context may include VAL because VAL is past history at TEST time.'
]),encoding='utf-8')
ZIP=WORK/'SASRec_serving_package.zip'; tmp=Path(str(ZIP)+'.tmp')
if tmp.exists(): tmp.unlink()
pack=[(FINAL_MODEL,'model/sasrec_best_model.pt'),(FAISS_PATH,'index/items_ivfpq.faiss'),(ACTIVE_MAP,'mappings/active_item_map.parquet'),(SERVING_CFG,'sasrec_config.json'),(README,'README_SERVING.txt'),(ROOT/'metrics'/'train_history.json','metrics/train_history.json'),(ROOT/'metrics'/'val_full_index_metrics.json','metrics/val_full_index_metrics.json'),(ROOT/'metrics'/'test_full_index_metrics.json','metrics/test_full_index_metrics.json'),(ROOT/'metrics'/'coverage_summary.json','metrics/coverage_summary.json'),(ROOT/'metrics'/'summary.json','metrics/summary.json')]
with zipfile.ZipFile(tmp,'w',allowZip64=True) as z:
    for src,arc in pack:
        src=Path(src)
        if not src.exists(): print('⚠️ Missing, skip:',src); continue
        comp=zipfile.ZIP_DEFLATED if src.suffix.lower() in ['.json','.txt'] else zipfile.ZIP_STORED; z.write(src,arcname=arc,compress_type=comp)
os.replace(tmp,ZIP); atomic_json({'state':'COMPLETE','serving_zip':str(ZIP),'completed_at':datetime.now().isoformat()},ROOT/'_COMPLETE.json')
print('✅ Serving ZIP:',ZIP); print(f'ZIP size={ZIP.stat().st_size/2**30:.2f}GB'); print('🎉 SASREC PIPELINE COMPLETE'); resources()